# Low water crossing risk, San Antonio: full pipeline

One notebook, top to bottom: **Kernel → Restart Kernel and Run All Cells**.

| Part | What it does | Needs internet |
| --- | --- | --- |
| 1 | Load and clean the city and state files | no |
| 2 | Join inlets, drainage projects and sewer overflows to each crossing | no |
| 3 | Live USGS stream gauges | yes (skipped if offline) |
| 4 | TxDOT traffic counts | yes (skipped if offline) |
| 5 | Risk score (z-scores), k-means clusters, priority score | no |
| 6 | Check against fire department rescue records | no |
| 7 | Save `crossings.csv` and the risk map | no |
| 8 | Charts for the slides | no |

**Inputs:** the downloaded CSVs in `data/raw/`. **Outputs:** `data/processed/crossings.csv` (used by `app.py`), `risk_map.html`, and PNG charts in `data/processed/figures/`.

In [ ]:
%pip install -q pyproj folium scipy scikit-learn seaborn requests

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from pyproj import Transformer

RAW = Path("data/raw")
OUT = Path("data/processed"); OUT.mkdir(parents=True, exist_ok=True)

def load(prefix):
    files = sorted(RAW.glob(prefix + "*.csv"))
    assert files, f"No file starting with {prefix!r} in {RAW.resolve()}"
    df = pd.read_csv(files[0], low_memory=False)
    df.columns = df.columns.str.strip()          # one inlet column name ends in a newline
    print(f"{prefix:42s} {df.shape}")
    return df

lwc     = load("LowWaterCrossings")
inlets  = load("StormInlets")
imp     = load("IMP_Drainage")
sso     = load("Texas_Commission_on_Environmental_Quality_-_Sanitary")

import requests
import folium
from branca.element import Element
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Part 1 · Load and clean

## 1. Coordinates

`x` and `y` are in **Texas State Plane South Central (EPSG:2278), in US feet**, so distances can be computed directly in feet. We also add latitude and longitude for the map.

In [ ]:
to_latlon = Transformer.from_crs(2278, 4326, always_xy=True)
lwc["lon"], lwc["lat"] = to_latlon.transform(lwc["x"].values, lwc["y"].values)

# sanity check: every crossing should fall inside the San Antonio area
assert lwc["lat"].between(29.1, 29.9).all() and lwc["lon"].between(-98.9, -98.1).all()
assert lwc["CartID"].is_unique
lwc[["Name", "CartID", "lat", "lon"]].head()

## 2. Clean the crossings

Pull out the creek name where the crossing's name includes one (for example *Rice Rd at Salado Creek*). Only some crossings name their creek; that limits the sewer overflow match in step 5.

In [ ]:
def creek_from_name(name):
    m = re.search(r"([A-Za-z]+\s+Creek)", name)
    return m.group(1).title() if m else None

lwc["Name"] = lwc["Name"].str.replace(r"\s+", " ", regex=True).str.strip()
lwc["creek"] = lwc["Name"].map(creek_from_name)
print(lwc["creek"].notna().sum(), "of", len(lwc), "crossings name a creek")
lwc["creek"].value_counts()

# Part 2 · Join everything to the crossings

## 3. Storm inlets near each crossing

Inlets carry street runoff into the drainage system. For each crossing we count **active** inlets within a radius, and how many of them are in **Poor or Failed** condition. Condition `TBD` means not yet inspected.

In [ ]:
INLET_RADIUS_FT = 1500     # about 460 m; a judgment call, try 1000 and 2500

inl = inlets.dropna(subset=["x", "y"]).copy()
inl = inl[(inl["x"] > 0) & (inl["y"] > 0)]
inl["Condition"] = inl["Condition"].str.strip().str.title()
inl["Status"] = inl["Status"].str.strip().str.title()
print(inl["Status"].value_counts().to_dict())
print(inl["Condition"].value_counts().to_dict())

active = inl[inl["Status"] == "Active"]
tree = cKDTree(active[["x", "y"]].values)
near = tree.query_ball_point(lwc[["x", "y"]].values, r=INLET_RADIUS_FT)

cond = active["Condition"].values
lwc["inlets_active"]       = [len(ix) for ix in near]
lwc["inlets_poor_failed"]  = [int(np.isin(cond[ix], ["Poor", "Failed"]).sum()) for ix in near]
lwc["inlets_uninspected"]  = [int((cond[ix] == "Tbd").sum()) for ix in near]
lwc["dist_nearest_inlet_ft"] = tree.query(lwc[["x", "y"]].values)[0].round(0)
lwc[["Name", "inlets_active", "inlets_poor_failed", "inlets_uninspected", "dist_nearest_inlet_ft"]].describe()

## 4. Drainage maintenance projects (IMP) near each crossing

The IMP Drainage file is the city's **five-year plan** (fiscal years 2026 to 2030) for creek debris removal and channel restoration. `INSIDE_X/Y` is a point on each project's creek segment. Most date columns are 80%+ empty, so we use the fiscal year instead.

In [ ]:
IMP_RADIUS_FT = 2640       # half a mile; project segments are long

imp_c = imp.dropna(subset=["INSIDE_X", "INSIDE_Y"]).copy()
imp_c["Project Type"] = imp_c["Project Type"].str.strip()
imp_c["Name"] = imp_c["Name"].str.strip().str.title()
print(imp_c["Project Type"].value_counts().to_dict(), "| fiscal years:", sorted(imp_c["Fiscal Year"].unique()))

itree = cKDTree(imp_c[["INSIDE_X", "INSIDE_Y"]].values)
inear = itree.query_ball_point(lwc[["x", "y"]].values, r=IMP_RADIUS_FT)
fy, cost, ptype = imp_c["Fiscal Year"].values, imp_c["Estimated Total Cost"].fillna(0).values, imp_c["Project Type"].values

lwc["imp_projects_nearby"]  = [len(ix) for ix in inear]
lwc["imp_restoration_nearby"] = [int((ptype[ix] == "Restoration").sum()) for ix in inear]
lwc["imp_next_fiscal_year"] = [int(fy[ix].min()) if len(ix) else np.nan for ix in inear]
lwc["imp_planned_cost"]     = [round(float(cost[ix].sum())) for ix in inear]
lwc["dist_nearest_imp_ft"]  = itree.query(lwc[["x", "y"]].values)[0].round(0)
print((lwc["imp_projects_nearby"] > 0).sum(), "crossings have planned drainage work within half a mile")

## 5. Sewer overflows on the same creek

The TCEQ file covers all of Texas and has **no coordinates**. We keep Bexar County, standardize the receiving water body (it is typed many ways: *SALADO CREEK*, *salado creek*, *1906 LEON CREEK*) and count overflows into each crossing's creek since 2021. *Infiltration and inflow* is the cause most tied to heavy rain.

In [ ]:
b = sso[sso["County"].str.strip().str.upper() == "BEXAR"].copy()
b["start"] = pd.to_datetime(b["Start Date"], format="%b %d, %Y", errors="coerce")
b["gallons"] = pd.to_numeric(b["Amount"].str.replace(",", ""), errors="coerce")
b["cause"] = b["Cause"].str.strip().str.upper()
b["water"] = (b["Receiving Water Body"].fillna("").str.upper()
              .str.replace(r"^\d+\s+", "", regex=True).str.strip()
              .map(lambda s: creek_from_name(s.title()) if "CREEK" in s else None))
b = b[b["start"] >= "2021-01-01"]
print(len(b), "Bexar overflows since 2021 |", b["start"].isna().sum(), "unparsed dates")
print(b["water"].value_counts().head(12).to_dict())

by_creek = b.dropna(subset=["water"]).groupby("water").agg(
    sso_same_creek=("Incident Number", "count"),
    sso_rain_related=("cause", lambda c: int((c == "INFILTRATION AND INFLOW").sum())),
    sso_gallons=("gallons", "sum"))
lwc = lwc.merge(by_creek, left_on="creek", right_index=True, how="left", validate="many_to_one")
for c in ["sso_same_creek", "sso_rain_related", "sso_gallons"]:
    lwc[c] = lwc[c].fillna(0).astype(int)
lwc.loc[lwc["creek"].notna(), ["Name", "creek", "sso_same_creek", "sso_rain_related"]]

# Part 3 · Live USGS stream gauges

The latest water level at every active USGS gauge in Bexar County, and each crossing's nearest gauge. USGS uses `-999999` for "no value", and some gauges haven't reported in years, so readings older than a day are kept as gauges but their number is dropped. If USGS can't be reached, this part is skipped and everything else still runs.

In [ ]:
USGS_URL = ("https://waterservices.usgs.gov/nwis/iv/?format=json"
            "&countyCd=48029&parameterCd=00065&siteStatus=active")   # Bexar County, gage height in feet
try:
    ts = requests.get(USGS_URL, timeout=60).json()["value"]["timeSeries"]
    gauges = pd.DataFrame([{
        "gauge_name": t["sourceInfo"]["siteName"],
        "g_lat": float(t["sourceInfo"]["geoLocation"]["geogLocation"]["latitude"]),
        "g_lon": float(t["sourceInfo"]["geoLocation"]["geogLocation"]["longitude"]),
        "gage_height_ft": float(t["values"][0]["value"][-1]["value"]),
        "gage_time": t["values"][0]["value"][-1]["dateTime"],
    } for t in ts if t["values"][0]["value"]])
    gauges["ts"] = pd.to_datetime(gauges["gage_time"], utc=True)
    stale = (gauges["gage_height_ft"] <= -999) | (gauges["ts"] < pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=1))
    gauges.loc[stale, "gage_height_ft"] = np.nan
    gauges["reporting"] = ~stale
    print(f"{len(gauges)} active gauges, {int(gauges['reporting'].sum())} with a current reading")
except Exception as e:
    gauges = pd.DataFrame()
    print("Could not reach USGS, skipping gauges:", e)

In [ ]:
if not gauges.empty:
    to_feet = Transformer.from_crs(4326, 2278, always_xy=True)       # lat/lon -> same feet grid as the city data
    gx, gy = to_feet.transform(gauges["g_lon"].values, gauges["g_lat"].values)
    dist, idx = cKDTree(np.column_stack([gx, gy])).query(lwc[["x", "y"]].values)
    near_g = gauges.iloc[idx].reset_index(drop=True)
    lwc["nearest_gauge"] = near_g["gauge_name"].values
    lwc["gauge_dist_mi"] = (dist / 5280).round(2)
    lwc["gauge_height_ft"] = near_g["gage_height_ft"].values
    lwc["gauge_reading_time"] = near_g["gage_time"].values
    lwc["gauge_reporting"] = near_g["reporting"].values
    within = lwc["gauge_dist_mi"] <= 1
    print(f"{within.sum()} crossings have a gauge within 1 mile; {(within & ~lwc['gauge_reporting']).sum()} of those gauges have no current reading")

# Part 4 · TxDOT traffic counts

How many vehicles pass each crossing (annual average daily traffic). Only counts within 1,500 ft are used. If TxDOT can't be reached, this part is skipped.

The AADT Annuals layer mostly covers state roads. If few crossings get a count, change the service name in `TXDOT` to `TxDOT_AADT_Urbans_2021`, which includes city streets, and rerun from here.

In [ ]:
TXDOT = ("https://services.arcgis.com/KTcxiTD9dsQw4r7Z/arcgis/rest/services/"
         "TxDOT_AADT_Annuals_%28Public_View%29/FeatureServer/0/query")
BEXAR_BOX = "-98.81,29.11,-98.11,29.76"          # lon/lat box around Bexar County

def get_traffic():
    rows, offset = [], 0
    while True:                                   # the service returns at most ~2,000 rows per page
        r = requests.get(TXDOT, timeout=60, params={
            "where": "1=1", "geometry": BEXAR_BOX, "geometryType": "esriGeometryEnvelope",
            "inSR": 4326, "spatialRel": "esriSpatialRelIntersects", "outFields": "*",
            "outSR": 4326, "f": "json", "resultOffset": offset, "resultRecordCount": 2000}).json()
        if "error" in r:
            raise RuntimeError(r["error"])
        feats = r.get("features", [])
        rows += [{**f["attributes"], "t_lon": f["geometry"]["x"], "t_lat": f["geometry"]["y"]} for f in feats]
        if not r.get("exceededTransferLimit") or not feats:
            return pd.DataFrame(rows)
        offset += len(feats)

try:
    traffic_raw = get_traffic()
    print(len(traffic_raw), "traffic count stations")
except Exception as e:
    traffic_raw = pd.DataFrame()
    print("Could not reach TxDOT:", e)
    print("Fallback: download 'TxDOT AADT Annuals' as CSV from gis-txdot.opendata.arcgis.com into data/raw/, then:")
    print("  traffic_raw = pd.read_csv(next(RAW.glob('*AADT*.csv')))   # needs lat/lon or x/y columns")
traffic_raw.columns.tolist()[:40] if not traffic_raw.empty else None

In [ ]:
def pick_aadt(df):
    """The newest AADT column. Newer layers have AADT_RPT_QTY; older ones have one column per year, e.g. F2023_TRAF."""
    for name in ["AADT_RPT_QTY", "AADT", "AADT_CUR"]:
        if name in df.columns:
            return name
    years = {c: int(m.group(0)) for c in df.columns
             if re.search(r"TRAF|AADT", c, re.I) and (m := re.search(r"(19|20)\d{2}", c))}
    if years:
        return max(years, key=years.get)
    raise KeyError(f"No AADT column found in: {df.columns.tolist()}")

if not traffic_raw.empty:
    AADT_COL = pick_aadt(traffic_raw)
    traffic = traffic_raw[["t_lon", "t_lat", AADT_COL]].rename(columns={AADT_COL: "aadt"})
    traffic["aadt"] = pd.to_numeric(traffic["aadt"], errors="coerce")
    traffic = traffic[traffic["aadt"] > 0].dropna()
    print(f"Using column {AADT_COL!r}: {len(traffic)} stations with a count, median {traffic['aadt'].median():,.0f} vehicles/day")

In [ ]:
TRAFFIC_RADIUS_FT = 1500    # a count farther away is probably on a different road

if not traffic_raw.empty:
    to_feet = Transformer.from_crs(4326, 2278, always_xy=True)
    tx, ty = to_feet.transform(traffic["t_lon"].values, traffic["t_lat"].values)
    dist, idx = cKDTree(np.column_stack([tx, ty])).query(lwc[["x", "y"]].values)
    lwc["aadt_dist_ft"] = dist.round(0)
    lwc["aadt_nearby"] = np.where(dist <= TRAFFIC_RADIUS_FT, traffic["aadt"].values[idx], np.nan)
    have = lwc["aadt_nearby"].notna()
    print(f"{have.sum()} of {len(lwc)} crossings have a traffic count within {TRAFFIC_RADIUS_FT:,} ft")
    print(lwc.loc[have, "aadt_nearby"].describe().round(0))

## 6. Check the merged table

One row per crossing, no rows lost, no missing values in the features.

In [ ]:
features = ["inlets_active", "inlets_poor_failed", "inlets_uninspected", "dist_nearest_inlet_ft",
            "imp_projects_nearby", "imp_restoration_nearby", "imp_planned_cost", "dist_nearest_imp_ft",
            "sso_same_creek", "sso_rain_related"]
assert len(lwc) == 156 and lwc["CartID"].is_unique
assert lwc[features].notna().all().all()
lwc[features].describe().round(1).T

# Part 5 · Risk score, clusters and priority

The score is deliberately simple so it can be explained on a slide. Each feature is converted to a z-score (how unusual it is compared with other crossings), then:

- **raises risk:** poor or failed inlets nearby, uninspected inlets nearby, distance to the nearest inlet, sewer overflows on the same creek
- **lowers risk:** more active inlets nearby, planned drainage work within half a mile

It ranks crossings by warning signs in the data. It is **not** a flood probability.

In [ ]:
z = pd.DataFrame(StandardScaler().fit_transform(lwc[features]), columns=features, index=lwc.index)
lwc["risk_score"] = (z["inlets_poor_failed"] + 0.5 * z["inlets_uninspected"] + z["dist_nearest_inlet_ft"]
                     + z["sso_same_creek"] - z["inlets_active"] - 0.5 * z["imp_projects_nearby"]).round(2)
lwc["risk_rank"] = lwc["risk_score"].rank(ascending=False, method="first").astype(int)

km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(z)
lwc["cluster"] = km.labels_
print("Average features per cluster (name them from this):")
lwc.groupby("cluster")[features + ["risk_score"]].mean().round(1).T

In [ ]:
lwc.nsmallest(10, "risk_rank")[["risk_rank", "Name", "risk_score", "inlets_active",
                                  "inlets_poor_failed", "dist_nearest_inlet_ft", "imp_projects_nearby", "sso_same_creek"]]

## Priority score: drainage risk + traffic exposure

The drainage risk score stays as it is. Traffic is added as exposure: how many people pass through. Crossings with no traffic count nearby are treated as average.

In [ ]:
# The drainage risk score stays as it is. Traffic is added as exposure: how many people pass through.
# Traffic counts are very skewed (a few roads carry 100,000+ cars), so we take the log before the z-score.
# Crossings with no count nearby get 0 (average), so missing data neither raises nor lowers them.
TRAFFIC_WEIGHT = 1.0

if "aadt_nearby" in lwc:
    logt = np.log1p(lwc["aadt_nearby"])
    lwc["traffic_z"] = ((logt - logt.mean()) / logt.std(ddof=0)).fillna(0).round(2)
    lwc["priority_score"] = (lwc["risk_score"] + TRAFFIC_WEIGHT * lwc["traffic_z"]).round(2)
    lwc["priority_rank"] = lwc["priority_score"].rank(ascending=False, method="first").astype(int)
    print(lwc.nsmallest(10, "priority_rank")[["priority_rank", "risk_rank", "Name", "aadt_nearby", "risk_score", "traffic_z"]])

# Part 6 · Check against real rescues

The San Antonio Fire Department's 10 locations with the most high-water rescues, 2015–2025 (released May 2026). The department noted the list isn't only low water crossings. Ten sites can't train a model, but they show how well each ranking matches reality.

In [ ]:
SAFD = {"Old Seguin at Salado Creek": 46, "Old OConnor N of Lookout Rd": 31, "HollyHock 600 W of Babcock": 20,
        "Pinn Rd 1320 ft S of West Commerce": 14, "Spencer Ln E of Balcones": 13, "Ira Lee N of Austin Hwy": 12,
        "Sleepy Hollow at Sunburst": 10, "Gibbs Sprawl at Rosillo Creek": 10,
        "W Commerce between Pinn Rd and Military": 9, "North Loop 150 ft SE of W North Loop Rd": 8}
lwc["safd_rescues_2015_2025"] = lwc["Name"].map(SAFD).fillna(0).astype(int)
check = lwc[lwc["safd_rescues_2015_2025"] > 0].sort_values("safd_rescues_2015_2025", ascending=False)
cols_check = ["Name", "safd_rescues_2015_2025", "risk_rank"] + (["aadt_nearby", "priority_rank"] if "priority_rank" in lwc else [])
print(check[cols_check].to_string(index=False))
for col in ["risk_rank", "priority_rank"]:
    if col in lwc:
        print(f"{col:14s}: median rank of the 10 rescue sites = {check[col].median():.0f} · in top 40: {(check[col] <= 40).sum()} of 10")

# Part 7 · Save and map

In [ ]:
save_cols = (["CartID", "Name", "creek", "lat", "lon", "x", "y"] + features +
             ["imp_next_fiscal_year", "risk_score", "risk_rank", "cluster"] +
             [c for c in ["nearest_gauge", "gauge_dist_mi", "gauge_height_ft", "gauge_reading_time", "gauge_reporting",
                          "aadt_nearby", "aadt_dist_ft", "traffic_z", "priority_score", "priority_rank",
                          "safd_rescues_2015_2025"] if c in lwc])
lwc[save_cols].sort_values("risk_rank").to_csv(OUT / "crossings.csv", index=False)
print("Saved", OUT / "crossings.csv", lwc[save_cols].shape)

In [ ]:
TIERS = [("Top 10: check first", 10, "#A32D2D", 9), ("Next 30: check soon", 40, "#EF9F27", 6),
         ("Lower priority", 999, "#888780", 4)]
tier = lambda rank: next((l, c, r) for l, m, c, r in TIERS if rank <= m)

m = folium.Map(location=[lwc["lat"].mean(), lwc["lon"].mean()], zoom_start=10)
for r in lwc.sort_values("risk_rank", ascending=False).itertuples():   # top-ranked drawn last, on top
    label, color, radius = tier(r.risk_rank)
    extra = ""
    if "aadt_nearby" in lwc and pd.notna(r.aadt_nearby):
        extra += f"<br>Traffic: {r.aadt_nearby:,.0f} vehicles/day"
    if r.safd_rescues_2015_2025:
        extra += f"<br><b>SAFD rescues 2015–2025: {r.safd_rescues_2015_2025}</b>"
    folium.CircleMarker(
        [r.lat, r.lon], radius=radius, color="white", weight=1, fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=(f"<b>#{r.risk_rank} {r.Name}</b><br>{label}<br>"
                 f"Active inlets nearby: {r.inlets_active} · poor/failed: {r.inlets_poor_failed}<br>"
                 f"Planned drainage projects: {r.imp_projects_nearby} · overflows on creek: {r.sso_same_creek}{extra}")
    ).add_to(m)
if not gauges.empty:
    for g in gauges.itertuples():
        reading = f"{g.gage_height_ft} ft at {g.gage_time[:16].replace('T', ' ')}" if g.reporting else "no current reading"
        folium.RegularPolygonMarker([g.g_lat, g.g_lon], number_of_sides=4, radius=5, color="#185FA5",
                                    fill=True, fill_opacity=0.9, tooltip=f"USGS: {g.gauge_name}<br>{reading}").add_to(m)

dot = lambda c: f'<span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:{c};margin-right:6px;vertical-align:middle"></span>'
rows = "".join(f"<div>{dot(c)}{l}</div>" for l, _, c, _ in TIERS)
if not gauges.empty:
    rows += ('<div><span style="display:inline-block;width:10px;height:10px;background:#185FA5;'
             'margin:0 8px 0 1px;vertical-align:middle"></span>USGS stream gauge</div>')
m.get_root().html.add_child(Element(
    '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:10px 14px;'
    'border-radius:8px;box-shadow:0 1px 4px rgba(0,0,0,.25);font:13px sans-serif;line-height:1.8">'
    f'<b>Low water crossings by risk rank</b>{rows}</div>'))
m.save(OUT / "risk_map.html"); m

# Part 8 · Charts for the slides

Saved as PNGs in `data/processed/figures/`.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="talk")

WEIGHTS = {"inlets_poor_failed": 1, "inlets_uninspected": 0.5, "dist_nearest_inlet_ft": 1,
           "sso_same_creek": 1, "inlets_active": -1, "imp_projects_nearby": -0.5}
LABELS = {"inlets_poor_failed": "Poor/failed inlets", "inlets_uninspected": "Uninspected inlets",
          "dist_nearest_inlet_ft": "Distance to inlet", "sso_same_creek": "Creek overflows",
          "inlets_active": "Active inlets", "imp_projects_nearby": "Planned projects"}
short = lambda n, k=30: n if len(n) <= k else n[:k - 1] + "…"
CLUSTER_COLORS = ["#378ADD", "#1D9E75", "#D85A30", "#888780", "#7F77DD", "#BA7517"]
PAL = {f"Cluster {k}": CLUSTER_COLORS[k % len(CLUSTER_COLORS)] for k in sorted(lwc["cluster"].unique())}  # same color per cluster everywhere

zf = pd.DataFrame(StandardScaler().fit_transform(lwc[features]), columns=features, index=lwc.index)
contrib = pd.DataFrame({LABELS[c]: zf[c] * w for c, w in WEIGHTS.items()})   # each feature's push on the score
contrib.index = lwc["Name"]

## Z-scores by feature
Each dot is one crossing; 0 is the average of all 156.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
long = zf[list(WEIGHTS)].rename(columns=LABELS).melt(var_name="Feature", value_name="z-score")
sns.stripplot(data=long, y="Feature", x="z-score", alpha=0.45, jitter=0.25, color="#46566A", ax=ax)
sns.boxplot(data=long, y="Feature", x="z-score", showfliers=False, width=0.5,
            boxprops={"facecolor": "none"}, ax=ax)
ax.axvline(0, color="#A32D2D", lw=1.5, ls="--")
ax.set_ylabel("")
ax.set_title("Z-scores: how unusual each crossing is, feature by feature\n(0 = average of all 156; each dot is one crossing)")
plt.tight_layout(); plt.savefig(FIG / "zscores_by_feature.png", dpi=150); plt.show()

## Why the top 10 rank high
Each colored segment is one feature's weighted z-score; they add up to the score.

In [ ]:
top = lwc.nsmallest(10, "risk_rank")
c = contrib.loc[top["Name"]]
c.index = [f"#{r} {short(n)}" for r, n in zip(top["risk_rank"], top["Name"])]
c = c.iloc[::-1]                                   # rank 1 at the top
colors = dict(zip(c.columns, sns.color_palette("colorblind", len(c.columns))))
fig, ax = plt.subplots(figsize=(15, 8))
for sign in (1, -1):                               # right of zero raises risk, left lowers it
    left = np.zeros(len(c))
    for col in c.columns:
        vals = c[col].clip(lower=0) if sign == 1 else c[col].clip(upper=0)
        ax.barh(c.index, vals, left=left, color=colors[col], edgecolor="white", height=0.72,
                label=col if sign == 1 else None)
        left += vals.values
pos_end = c.clip(lower=0).sum(axis=1)
for i, (y, total) in enumerate(zip(c.index, top["risk_score"].iloc[::-1])):
    ax.text(pos_end.iloc[i] + 0.2, i, f"score {total:.1f}", va="center", fontsize=13, weight="bold")
ax.axvline(0, color="black", lw=1.2)
ax.set_xlim(c.clip(upper=0).sum(axis=1).min() - 0.5, pos_end.max() + 2.2)
ax.set_xlabel("← lowers risk          contribution to risk score (weighted z-score)          raises risk →")
ax.set_title("Why the top 10 rank high: each colored segment is one warning sign")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=12, title="Feature")
plt.tight_layout(); plt.savefig(FIG / "top10_contributions.png", dpi=150); plt.show()

## Top 20 heatmap

In [ ]:
top20 = lwc.nsmallest(20, "risk_rank")
heat = contrib.loc[top20["Name"]]
heat.index = [f"#{r} {short(n)}" for r, n in zip(top20["risk_rank"], top20["Name"])]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(heat, cmap="RdBu_r", center=0, vmin=-3, vmax=3, annot=True, fmt=".1f",
            annot_kws={"size": 11}, cbar_kws={"label": "Pushes risk down  ←  0  →  pushes risk up"}, ax=ax)
ax.set_title("Top 20 crossings: what drives each score\n(colors capped at ±3; numbers show the real value)")
ax.set_ylabel(""); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.savefig(FIG / "top20_heatmap.png", dpi=150); plt.show()

## K-means clusters on a 2D map
PCA squeezes the features onto 2 axes so the clusters can be drawn. Red rings mark the top 10.

In [ ]:
X = StandardScaler().fit_transform(lwc[features])
pca = PCA(n_components=2, random_state=0).fit(X)
pts = pd.DataFrame(pca.transform(X), columns=["PC1", "PC2"])
pts["Cluster"] = "Cluster " + lwc["cluster"].astype(str).values
order = sorted(pts["Cluster"].unique())
fig, ax = plt.subplots(figsize=(13, 8))
sns.scatterplot(data=pts, x="PC1", y="PC2", hue="Cluster", hue_order=order, palette=PAL,
                s=90, alpha=0.8, edgecolor="white", ax=ax)
topmask = lwc["risk_rank"].values <= 10                         # ring the top 10
ax.scatter(pts.loc[topmask, "PC1"], pts.loc[topmask, "PC2"], s=320, facecolors="none",
           edgecolors="#A32D2D", linewidths=2.2, label="Top 10 risk")
xmid = pts["PC1"].median()
for i in np.where(lwc["risk_rank"].values <= 3)[0]:
    p, rank, name = pts.iloc[i], lwc["risk_rank"].iloc[i], lwc["Name"].iloc[i]
    right = p.PC1 < xmid + 2                                     # put labels on the side with room
    ax.annotate(f"#{rank} {short(name, 34)}", (p.PC1, p.PC2), xytext=(12 if right else -12, -4),
                textcoords="offset points", ha="left" if right else "right", fontsize=12)
v = pca.explained_variance_ratio_
ax.set_xlabel(f"Component 1 ({v[0]:.0%} of the variation)"); ax.set_ylabel(f"Component 2 ({v[1]:.0%})")
ax.set_title("K-means clusters: 156 crossings squeezed onto 2 axes\n(nearby dots have similar warning signs)")
ax.margins(x=0.08, y=0.08)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=12)
plt.tight_layout(); plt.savefig(FIG / "kmeans_pca.png", dpi=150); plt.show()

## What makes each cluster different
Use this to name the clusters.

In [ ]:
prof = zf.groupby(lwc["cluster"])[list(WEIGHTS)].mean().rename(columns=LABELS)
sizes = lwc["cluster"].value_counts().sort_index()
prof.index = [f"Cluster {k} ({sizes[k]} crossings)" for k in prof.index]
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(prof, cmap="RdBu_r", center=0, vmin=-3, vmax=3, annot=True, fmt=".1f", ax=ax,
            cbar_kws={"label": "Avg z-score"})
ax.set_title("What makes each cluster different (average z-score per feature)")
ax.set_ylabel(""); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.savefig(FIG / "cluster_profiles.png", dpi=150); plt.show()

## Risk score within each cluster

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
d = lwc.assign(Cluster="Cluster " + lwc["cluster"].astype(str))
sns.boxplot(data=d, x="Cluster", y="risk_score", order=order, palette=PAL,
            hue="Cluster", legend=False, showfliers=False, boxprops={"alpha": 0.35}, ax=ax)
sns.stripplot(data=d, x="Cluster", y="risk_score", order=order, palette=PAL,
              hue="Cluster", legend=False, size=7, jitter=0.2, ax=ax)
ax.axhline(0, color="#46566A", lw=1, ls="--")
ax.set_ylabel("Risk score"); ax.set_xlabel("")
ax.set_title("Risk score within each cluster (each dot is one crossing)")
plt.tight_layout(); plt.savefig(FIG / "risk_by_cluster.png", dpi=150); plt.show()
print("Saved charts to", FIG)

## Notes
- **Storm channels aren't used:** the CSV has no locations. Download it as GeoJSON to add distance to the nearest channel.
- **Sewer overflows only reach crossings whose name includes a creek;** for the rest, 0 means "unknown", not "none".
- **Radii are judgment calls** (inlets 1,500 ft, drainage projects ½ mile, traffic 1,500 ft). Try other values and check whether the top 10 changes.
- **Gage height is the water level at the gauge in its own local measure,** not the water depth on the road.